# Credit Card Approval Prediction — Model Building

Trains and compares 4 classification algorithms:
Logistic Regression, Decision Tree, Random Forest, XGBoost (falls back to
Gradient Boosting if `xgboost` isn't installed), then saves the best one.

In [ ]:
import sys
sys.path.append('..')
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

try:
    from xgboost import XGBClassifier
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier

from preprocessing import build_dataset, FEATURE_COLS

## Load the preprocessed dataset

In [ ]:
X_scaled, y, encoders, scaler, df = build_dataset(
    app_path='../data/application_record.csv',
    credit_path='../data/credit_record.csv',
)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Train & evaluate all 4 models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08, random_state=42),
}

results = {}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    results[name] = {'accuracy': acc, 'roc_auc': auc}
    fitted[name] = model
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f}  ROC-AUC: {auc:.4f}')
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))


## Select and inspect the best model

In [ ]:
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best_model = fitted[best_name]
print('Best model:', best_name, results[best_name])

## Save the model + preprocessing artifacts

These are the exact files the Flask app (`app.py`) loads at startup.

In [ ]:
with open('../model/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../model/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
with open('../model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Saved.')